In [1]:
import csv
import json
import os
import sys
import types
from collections import Counter
from datetime import datetime, timedelta, timezone
from importlib.metadata import version
from pathlib import Path

import requests
from dotenv import load_dotenv

import cattr
import httpx._types as httpx_types

if not hasattr(httpx_types, "VerifyTypes"):
    httpx_types.VerifyTypes = bool | str | Path

if "pkg_resources" not in sys.modules:
    pkg_resources = types.ModuleType("pkg_resources")

    class _Distribution:
        def __init__(self, package_name: str):
            self.version = version(package_name)

    pkg_resources.get_distribution = _Distribution
    sys.modules["pkg_resources"] = pkg_resources

if not hasattr(cattr.converters.BaseConverter, "_unstructure_enum"):
    def _unstructure_enum(self, obj):
        return obj.value

    cattr.converters.BaseConverter._unstructure_enum = _unstructure_enum

import toloka.client as toloka
import toloka.client.project.template_builder as tb
from toloka.client.actions import SetSkillFromOutputField
from toloka.client.collectors import GoldenSet
from toloka.client.conditions import RuleConditionKey, TotalAnswersCount
from toloka.client.search_requests import UserSkillSearchRequest


In [2]:
load_dotenv()

TOKEN = os.getenv("TOLOKA_TOKEN")
ENVIRONMENT = "PRODUCTION"
TOLOKA_API_URL = "https://tasks.yandex.ru"
REVIEW_PROJECT_NAME = "[review] Проверка сегментации бабочек"
REVIEW_PROJECT_ID = "9620" # !!!
REVIEW_EXAM_SKILL_NAME = "review_exam_passed"
TARGET_PASSED_REVIEWERS = 50

review_interface_path = "data/review-interface.json"
review_instruction_path = "data/review-instruction.html"
review_exam_tsv_path = "data/review_project_exam.tsv"

toloka_client = toloka.TolokaClient(TOKEN, url=TOLOKA_API_URL)

## Проверка доступа

In [3]:
requester = toloka_client.get_requester()
print("public name:", getattr(requester, "public_name", None))
print("balance:", getattr(requester, "balance", None))


public name: {'EN': 'eraserhead@aiconf-2026'}
balance: 1491.4000


## Навык экзамена

In [4]:
existing_review_skill = next(toloka_client.get_skills(name=REVIEW_EXAM_SKILL_NAME), None)

if existing_review_skill is None:
    review_exam_skill = toloka_client.create_skill(
        hidden=True,
        name=REVIEW_EXAM_SKILL_NAME,
        private_comment="Процент правильных ответов на review exam",
    )
else:
    review_exam_skill = existing_review_skill

print("review_exam_skill_id:", review_exam_skill.id)


review_exam_skill_id: 16286


## Создание review-проекта

In [5]:
with open(review_interface_path, encoding="utf-8") as file:
    review_interface_config = json.load(file)

with open(review_instruction_path, encoding="utf-8") as file:
    review_project_instructions = file.read()

review_view_spec = toloka.project.TemplateBuilderViewSpec(
    config=tb.TemplateBuilder.structure(review_interface_config)
)

if REVIEW_PROJECT_ID:
    review_project = toloka_client.get_project(REVIEW_PROJECT_ID)
else:
    review_project = toloka.project.Project(
        public_name=REVIEW_PROJECT_NAME,
        public_description="Проект для проверки сегментации.",
        public_instructions=review_project_instructions,
        task_spec=toloka.project.task_spec.TaskSpec(
            input_spec={
                "image": toloka.project.field_spec.UrlSpec(required=True),
                "guides": toloka.project.field_spec.JsonSpec(required=False),
                "taxon": toloka.project.field_spec.StringSpec(required=False),
                "photo_id": toloka.project.field_spec.StringSpec(required=False),
                "hard": toloka.project.field_spec.StringSpec(required=False),
            },
            output_spec={
                "result": toloka.project.field_spec.StringSpec(required=True),
                "path": toloka.project.field_spec.JsonSpec(required=False),
            },
            view_spec=review_view_spec,
        ),
    )

    review_project = toloka_client.create_project(review_project)
review_project


Project(_unexpected={'public_instructions_price': Decimal('0.00'), 'service_type': 'UNDEFINED', 'project_types': [], 'content_types': [], 'owner': {'id': '4f61e1bd6deac48f1628b09918db2205', 'myself': True, 'company_id': '151'}}, public_name='[review] Проверка сегментации бабочек', public_description='Проект для проверки сегментации.', task_spec=TaskSpec(_unexpected={}, input_spec={'hard': StringSpec(_unexpected={}, required=False, hidden=False, min_length=None, max_length=None, allowed_values=None), 'image': UrlSpec(_unexpected={}, required=True, hidden=False), 'taxon': StringSpec(_unexpected={}, required=False, hidden=False, min_length=None, max_length=None, allowed_values=None), 'guides': JsonSpec(_unexpected={}, required=False, hidden=False), 'photo_id': StringSpec(_unexpected={}, required=False, hidden=False, min_length=None, max_length=None, allowed_values=None)}, output_spec={'path': JsonSpec(_unexpected={}, required=False, hidden=False), 'result': StringSpec(_unexpected={}, requ

## Создание экзаменационного пула внутри review-проекта

In [6]:
with open(review_exam_tsv_path, encoding="utf-8", newline="") as file:
    review_exam_rows = list(csv.DictReader(file, delimiter="	"))

review_exam_tasks_count = len(review_exam_rows)

review_exam_pool = toloka.pool.Pool(
    project_id=review_project.id,
    private_name=f"review_exam_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    may_contain_adult_content=False,
    reward_per_assignment=0.5,
    assignment_max_duration_seconds=10 * 60,
    will_expire=datetime.now(timezone.utc) + timedelta(days=30),
    auto_accept_solutions=True,
    defaults=toloka.pool.Pool.Defaults(
        default_overlap_for_new_task_suites=99,
    ),
    filter=(
        (toloka.filter.ClientType == "BROWSER")
    ) & (
        toloka.filter.Languages.in_("RU")
    ) & toloka.filter.FilterOr([
        toloka.filter.RegionByPhone.in_(225),
        toloka.filter.RegionByIp.in_(225),
    ]),
)

review_exam_pool.set_mixer_config(
    real_tasks_count=0,
    golden_tasks_count=review_exam_tasks_count,
    training_tasks_count=0,
    min_golden_tasks_count=review_exam_tasks_count,
    mix_tasks_in_creation_order=False,
    shuffle_tasks_in_task_suite=True,
)

review_exam_pool.quality_control.add_action(
    collector=GoldenSet(history_size=review_exam_tasks_count),
    conditions=[TotalAnswersCount >= review_exam_tasks_count],
    action=SetSkillFromOutputField(
        skill_id=review_exam_skill.id,
        from_field=RuleConditionKey("correct_answers_rate"),
    ),
)

review_exam_pool = toloka_client.create_pool(review_exam_pool)
print("project_id:", review_project.id)
print("pool_id:", review_exam_pool.id)

RetryError: HTTPSConnectionPool(host='tasks.yandex.ru', port=443): Max retries exceeded with url: /api/v1/pools (Caused by ResponseError('too many 500 error responses'))

## Загрузка заданий из review_project_exam.tsv

In [ ]:
review_exam_tasks = []

for row in review_exam_rows:
    raw_guides = row["INPUT:guides"]
    guides = None
    if raw_guides and raw_guides != "{}":
        guides = json.loads(raw_guides.replace('\,', ','))
        if isinstance(guides, dict):
            guides = [guides]

    review_exam_tasks.append(
        toloka.Task(
            pool_id=review_exam_pool.id,
            input_values={
                "image": row["INPUT:image"],
                "guides": guides,
                "taxon": row["INPUT:taxon"],
                "photo_id": row["INPUT:photo_id"],
                "hard": row["INPUT:hard"],
            },
            known_solutions=[
                toloka.task.BaseTask.KnownSolution(
                    output_values={"result": row["GOLDEN:result"]},
                )
            ],
            infinite_overlap=True,
        )
    )

review_exam_upload_result = toloka_client.create_tasks(review_exam_tasks, allow_defaults=True)
print("Заданий загружено:", len(review_exam_upload_result.items))


<>:7: SyntaxWarning: invalid escape sequence '\,'
<>:7: SyntaxWarning: invalid escape sequence '\,'
/var/folders/51/txrzn2x90ld5shrztyy9l3p00000gn/T/ipykernel_23769/844681532.py:7: SyntaxWarning: invalid escape sequence '\,'
  guides = json.loads(raw_guides.replace('\,', ','))
100%|██████████| 100/100 [00:02<00:00, 46.45it/s]


Заданий загружено: 10


## Идем в ЯЗ, жмем "Отправить на модерацию"...

## Проверка статуса проекта и пула

In [ ]:
fresh_project = toloka_client.get_project(review_project.id)
fresh_pool = toloka_client.get_pool(review_exam_pool.id)

print("project_status:", getattr(fresh_project, "status", None))
print("pool_status:", getattr(fresh_pool, "status", None))
print("last_close_reason:", getattr(fresh_pool, "last_close_reason", None))


project_status: ProjectStatus.ACTIVE
pool_status: Status.CLOSED
last_close_reason: None


## Мониторинг прогресса по навыку review_exam_passed

In [ ]:
user_skills = list(
    toloka_client.get_user_skills(
        UserSkillSearchRequest(skill_id=review_exam_skill.id)
    )
)

best_scores = {}
for user_skill in user_skills:
    user_id = getattr(user_skill, "user_id", None)
    exact_value = getattr(user_skill, "exact_value", None)
    if user_id is None or exact_value is None:
        continue
    best_scores[user_id] = max(exact_value, best_scores.get(user_id, exact_value))

score_distribution = Counter(best_scores.values())
passed_user_ids = [user_id for user_id, score in best_scores.items() if score == 100]

print("Уникальных исполнителей с навыком:", len(best_scores))
print("Исполнителей с результатом 100:", len(passed_user_ids))

passed_user_ids[:10]


Уникальных исполнителей с навыком: 205
Исполнителей с результатом 100: 100


['0fa81c631c7ddf6866d0e74b445def47',
 'c2191c41ff6320a67d66816e9c2f0df5',
 '27e60592fe902b5cecc3299c9a95014b',
 '25afea86c1456290ca3efc6253f437d7',
 'bc7af50ccd5a510a014b4465e414a104',
 'aa23658d95c52fc29caa918edc4dcd91',
 '50bd0c2732be17ab996df426675a38b1',
 'f2eb83b9be3075d3f1c82c9870effc20',
 'bef4363da24be76bfc9363f877106cfe',
 'faf537457a0791b578265c5ab141770d']

In [ ]:
toloka_client.close_pool(review_exam_pool.id)

Pool(_unexpected={'may_contain_nda': False, 'eligible_audience': 'ANY'}, project_id='9620', private_name='review_exam_20260418_203012', may_contain_adult_content=False, reward_per_assignment=1.0, assignment_max_duration_seconds=600, defaults=Pool.Defaults(_unexpected={}, default_overlap_for_new_task_suites=99, default_overlap_for_new_tasks=None), will_expire=datetime.datetime(2026, 5, 18, 17, 30, 12, 834000, tzinfo=datetime.timezone.utc), private_comment=None, public_description=None, public_instructions=None, auto_close_after_complete_delay_seconds=0, dynamic_pricing_config=None, auto_accept_solutions=True, auto_accept_period_day=21, assignments_issuing_config=Pool.AssignmentsIssuingConfig(_unexpected={}, issue_task_suites_in_creation_order=False), priority=0, filter=FilterAnd(_unexpected={}, and_=[FilterOr(_unexpected={}, or_=[ClientType(_unexpected={}, operator=<IdentityOperator.EQ: 'EQ'>, value=<ClientType.BROWSER: 'BROWSER'>)]), FilterOr(_unexpected={}, or_=[Languages(_unexpected=